# Movie 8 — Pipeline complet 4 phases

**Phase 1** — Base solide : config movie6bis + head+tail calibré + SVM  
**Phase 2** — Tuning epochs & LR (grid search)  
**Phase 3** — DeBERTa-v3-base  
**Phase 4** — Vote majoritaire pondéré (soft voting) + 4 fichiers de soumission  

Objectif : dépasser 92.647 F1 sur le test plateforme.

## 0. Installs & imports

In [8]:
!pip install -q transformers datasets accelerate scikit-learn

from pathlib import Path
import os, random, warnings, gc, itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

import torch
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration globale

In [10]:
# ===== Chemins — adapte si besoin =====
DATA_DIR   = Path("/content/drive/MyDrive/projet tal/movies1000/movies1000")
TEST_FILE  = Path("/content/drive/MyDrive/projet tal/test.txt")
OUTPUT_DIR = Path("/content/drive/MyDrive/projet tal/movie8_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Labels =====
label2id = {"N": 0, "P": 1}
id2label  = {0: "N", 1: "P"}

# ===== Seed =====
SEED = 42

# ===== SVM (config movie6bis) =====
SVM_NGRAM_RANGE = (1, 2)
SVM_MIN_DF      = 3
SVM_MAX_DF      = 0.95
SVM_SUBLINEAR_TF = True
SVM_C           = 2.0

# ===== Head+tail calibré =====
# Ratio de tokens pris en début vs fin de séquence
# 0.5 = 50% début / 50% fin  (moins agressif que movie7)
HEAD_RATIO   = 0.5
MAX_LENGTH   = 512

# ===== RoBERTa (config movie6bis comme base) =====
ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
ROBERTA_BASE_EPOCHS = 2
ROBERTA_BASE_LR     = 2e-5
ROBERTA_BATCH_TRAIN = 8
ROBERTA_BATCH_EVAL  = 16
ROBERTA_WEIGHT_DECAY = 0.01

# ===== DeBERTa =====
DEBERTA_MODEL = "microsoft/deberta-v3-base"
DEBERTA_BASE_EPOCHS = 3
DEBERTA_BASE_LR     = 2e-5
DEBERTA_BATCH_TRAIN = 8
DEBERTA_BATCH_EVAL  = 16
DEBERTA_WEIGHT_DECAY = 0.01

# ===== Divers =====
MAKE_TEST_SUBMISSIONS = True
SAVE_TRAINED_MODELS   = True

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("Config OK — OUTPUT_DIR:", OUTPUT_DIR)

Config OK — OUTPUT_DIR: /content/drive/MyDrive/projet tal/movie8_outputs


## 2. Chargement des données

In [4]:
def load_movies_from_folder(data_dir: Path) -> pd.DataFrame:
    rows = []
    for label_folder, label in [("pos", "P"), ("neg", "N")]:
        folder = data_dir / label_folder
        if not folder.exists():
            continue
        for file_path in folder.glob("*.txt"):
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            rows.append({"doc_id": file_path.name, "label": label, "text": text})
    if not rows:
        raise ValueError("Aucun fichier trouvé. Vérifie DATA_DIR.")
    return pd.DataFrame(rows).sort_values("doc_id").reset_index(drop=True)

def load_test_reviews(test_file: Path) -> pd.DataFrame | None:
    if not test_file.exists():
        print("Fichier test introuvable :", test_file)
        return None
    lines = test_file.read_text(encoding="utf-8", errors="ignore").splitlines()
    lines = [l.strip() for l in lines if l.strip()]
    return pd.DataFrame({"row_id": np.arange(len(lines)), "text": lines})

df      = load_movies_from_folder(DATA_DIR)
test_df = load_test_reviews(TEST_FILE)

print("Train complet :", df.shape)
print(df["label"].value_counts())
if test_df is not None:
    print("Test :", test_df.shape)

Fichier test introuvable : /content/drive/MyDrive/projet tal/test.txt
Train complet : (2000, 3)
label
N    1000
P    1000
Name: count, dtype: int64


## 3. Split validation

In [5]:
train_df, valid_df = train_test_split(
    df[["doc_id", "label", "text"]].copy(),
    test_size=0.2, random_state=SEED, stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
print("Train:", train_df.shape, "Valid:", valid_df.shape)

Train: (1600, 3) Valid: (400, 3)


## 4. Utilitaires communs

In [6]:
def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    y_true = np.array([id2label[int(x)] for x in labels])
    y_pred = np.array([id2label[int(x)] for x in preds])
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="P"),
        "recall":    recall_score(y_true, y_pred, pos_label="P"),
        "f1":        f1_score(y_true, y_pred, pos_label="P"),
    }

def print_report(y_true, y_pred, title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)
    print(classification_report(y_true, y_pred, digits=4))
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="P"),
        "recall":    recall_score(y_true, y_pred, pos_label="P"),
        "f1":        f1_score(y_true, y_pred, pos_label="P"),
    }

def save_submission(labels, path: Path):
    sub = pd.DataFrame({"label": labels})
    sub.to_csv(path, index=False)
    print(f"✓ Soumission : {path.name}  |  {dict(sub['label'].value_counts())}")
    return sub

# Stockage des résultats de toutes les phases
RESULTS = {}  # {nom: {accuracy, precision, recall, f1, probs_valid, pred_valid}}
print("Utilitaires OK")

Utilitaires OK


---
# PHASE 1 — Base solide
Config movie6bis : SVM + RoBERTa avec head+tail calibré (HEAD_RATIO=0.5)

### 1a. SVM optimisé

In [7]:
def build_svm():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=SVM_NGRAM_RANGE,
            min_df=SVM_MIN_DF,
            max_df=SVM_MAX_DF,
            sublinear_tf=SVM_SUBLINEAR_TF,
        )),
        ("clf", LinearSVC(C=SVM_C, max_iter=2000))
    ])

svm_model = build_svm()
svm_model.fit(train_df["text"], train_df["label"])
valid_pred_svm = svm_model.predict(valid_df["text"])

# SVM n'a pas de probas natives → on utilise decision_function
svm_scores = svm_model.decision_function(valid_df["text"])  # shape (N,)
# Convertir en pseudo-probas softmax 2-classes [P(N), P(P)]
svm_probs_valid = softmax_np(np.column_stack([-svm_scores, svm_scores]))

svm_metrics = print_report(valid_df["label"], valid_pred_svm, "SVM (validation)")
RESULTS["SVM"] = {**svm_metrics, "probs_valid": svm_probs_valid, "pred_valid": valid_pred_svm}


  SVM (validation)
              precision    recall  f1-score   support

           N     0.9062    0.8700    0.8878       200
           P     0.8750    0.9100    0.8922       200

    accuracy                         0.8900       400
   macro avg     0.8906    0.8900    0.8900       400
weighted avg     0.8906    0.8900    0.8900       400



### 1b. Head+tail : fonction de tokenisation calibrée

Pour les textes longs (>512 tokens), on prend HEAD_RATIO du budget en début et (1-HEAD_RATIO) en fin.  
HEAD_RATIO=0.5 → 256 tokens début + 256 tokens fin — moins agressif que movie7.

In [11]:
def make_head_tail_tokenize_fn(tokenizer, max_length=MAX_LENGTH, head_ratio=HEAD_RATIO):
    """
    Retourne une fonction de tokenisation head+tail calibrée.
    head_ratio=0.5 → budget partagé à égalité début/fin.
    """
    # Réserver 2 tokens pour [CLS] et [SEP]
    content_len = max_length - 2
    n_head = int(content_len * head_ratio)
    n_tail = content_len - n_head

    def tokenize_fn(batch):
        all_input_ids, all_attention_mask = [], []

        for text in batch["text"]:
            # Tokeniser sans troncature pour avoir tous les tokens
            enc = tokenizer(
                text,
                add_special_tokens=False,
                truncation=False,
            )
            ids = enc["input_ids"]

            if len(ids) <= content_len:
                # Texte court : troncature standard
                selected = ids
            else:
                # Texte long : head + tail
                selected = ids[:n_head] + ids[-n_tail:]

            # Ajouter [CLS] et [SEP]
            cls_id = tokenizer.cls_token_id
            sep_id = tokenizer.sep_token_id
            if cls_id is None:  # pour DeBERTa qui utilise <s>
                cls_id = tokenizer.bos_token_id
            if sep_id is None:
                sep_id = tokenizer.eos_token_id

            final_ids = [cls_id] + selected + [sep_id]
            pad_len = max_length - len(final_ids)
            attention = [1] * len(final_ids) + [0] * pad_len
            final_ids = final_ids + [tokenizer.pad_token_id] * pad_len

            all_input_ids.append(final_ids)
            all_attention_mask.append(attention)

        return {"input_ids": all_input_ids, "attention_mask": all_attention_mask}

    return tokenize_fn

print(f"head+tail calibré : {int((MAX_LENGTH-2)*HEAD_RATIO)} tokens début + {int((MAX_LENGTH-2)*(1-HEAD_RATIO))} tokens fin")

head+tail calibré : 255 tokens début + 255 tokens fin


### 1c. Fonction d'entraînement transformer générique (réutilisée dans toutes les phases)

In [12]:
def run_transformer(
    model_name: str,
    run_name: str,
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    num_epochs: int = 2,
    learning_rate: float = 2e-5,
    weight_decay: float = 0.01,
    warmup_ratio: float = 0.06,
    batch_train: int = 8,
    batch_eval: int = 16,
    max_length: int = MAX_LENGTH,
    head_ratio: float = HEAD_RATIO,
    use_early_stopping: bool = True,
    early_stopping_patience: int = 2,
    save_model: bool = SAVE_TRAINED_MODELS,
) -> dict:
    """
    Entraîne un transformer avec head+tail et retourne les prédictions sur la validation.
    """
    set_seed()
    print(f"\n{'#'*70}")
    print(f"  {run_name}")
    print(f"  epochs={num_epochs} | lr={learning_rate} | wd={weight_decay} | warmup={warmup_ratio}")
    print(f"{'#'*70}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Datasets HF
    def make_hf(df):
        tmp = df[["text", "label"]].copy()
        tmp["label"] = tmp["label"].map(label2id).astype(int)
        return Dataset.from_pandas(tmp.rename(columns={"label": "label"}))

    hf_train = make_hf(train_df)
    hf_valid = make_hf(valid_df)

    # Tokenisation head+tail
    tokenize_fn = make_head_tail_tokenize_fn(tokenizer, max_length, head_ratio)
    tok_train = hf_train.map(tokenize_fn, batched=True, remove_columns=["text"])
    tok_valid = hf_valid.map(tokenize_fn, batched=True, remove_columns=["text"])

    # Supprimer colonnes parasites
    for ds in [tok_train, tok_valid]:
        for col in ["__index_level_0__"]:
            if col in ds.column_names:
                ds = ds.remove_columns([col])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    output_dir = str(OUTPUT_DIR / f"{run_name}_ckpt")

    callbacks = []
    if use_early_stopping:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_train,
        per_device_eval_batch_size=batch_eval,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        load_best_model_at_end=True,
        metric_for_best_model="f1",      # optimiser F1, pas accuracy
        greater_is_better=True,
        report_to="none",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_valid,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    trainer.train()

    # Prédictions validation
    pred_output = trainer.predict(tok_valid)
    logits = pred_output.predictions
    probs  = softmax_np(logits)
    pred_ids    = np.argmax(logits, axis=1)
    pred_labels = np.array([id2label[int(x)] for x in pred_ids])

    metrics = print_report(valid_df["label"], pred_labels, run_name)

    if save_model:
        save_dir = OUTPUT_DIR / f"{run_name}_best_model"
        trainer.save_model(str(save_dir))
        tokenizer.save_pretrained(str(save_dir))
        print(f"  Modèle sauvegardé : {save_dir}")

    # Libérer la VRAM
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        **metrics,
        "probs_valid": probs,
        "pred_valid": pred_labels,
        "trainer": trainer,
        "tokenizer": tokenizer,
    }

print("run_transformer() défini")

run_transformer() défini


### 1d. RoBERTa base (config movie6bis + head+tail calibré)

In [13]:
roberta_base_result = run_transformer(
    model_name=ROBERTA_MODEL,
    run_name="roberta_base_p1",
    train_df=train_df,
    valid_df=valid_df,
    num_epochs=ROBERTA_BASE_EPOCHS,
    learning_rate=ROBERTA_BASE_LR,
    weight_decay=ROBERTA_WEIGHT_DECAY,
    warmup_ratio=0.06,
    batch_train=ROBERTA_BATCH_TRAIN,
    batch_eval=ROBERTA_BATCH_EVAL,
    head_ratio=HEAD_RATIO,
    use_early_stopping=False,   # 2 epochs : pas besoin
)
RESULTS["RoBERTa_base"] = roberta_base_result


######################################################################
  roberta_base_p1
  epochs=2 | lr=2e-05 | wd=0.01 | warmup=0.06
######################################################################


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


  roberta_base_p1
              precision    recall  f1-score   support

           N     0.9624    0.8950    0.9275       200
           P     0.9019    0.9650    0.9324       200

    accuracy                         0.9300       400
   macro avg     0.9321    0.9300    0.9299       400
weighted avg     0.9321    0.9300    0.9299       400



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Modèle sauvegardé : /content/drive/MyDrive/projet tal/movie8_outputs/roberta_base_p1_best_model


### 1e. Résumé Phase 1

In [14]:
summary_p1 = pd.DataFrame([
    {"modèle": k, **{m: round(v,4) for m,v in RESULTS[k].items() if m in ["accuracy","precision","recall","f1"]}}
    for k in ["SVM", "RoBERTa_base"]
]).set_index("modèle")
print("\n=== Résumé Phase 1 ===")
display(summary_p1)
print("Baseline movie6bis visée : F1 ~ 0.926")


=== Résumé Phase 1 ===


,accuracy,precision,recall,f1
modèle,,,,
SVM,0.89,0.8750,0.910,0.8922
RoBERTa_base,0.93,0.9019,0.965,0.9324


Baseline movie6bis visée : F1 ~ 0.926


---
# PHASE 2 — Grid search : epochs × LR × weight_decay

On cherche la meilleure combinaison pour RoBERTa.  
**Early stopping sur F1 validation** — on ne laisse pas le modèle sur-entraîner.

In [15]:
# Grid de recherche — modifie si tu veux élargir
GRID_EPOCHS       = [2, 3, 4]
GRID_LR           = [2e-5, 3e-5]
GRID_WARMUP       = [0.06, 0.1]
GRID_WEIGHT_DECAY = [0.01, 0.1]

grid_results = []

for epochs, lr, warmup, wd in itertools.product(
    GRID_EPOCHS, GRID_LR, GRID_WARMUP, GRID_WEIGHT_DECAY
):
    run_name = f"roberta_grid_e{epochs}_lr{lr:.0e}_w{warmup}_wd{wd}"
    res = run_transformer(
        model_name=ROBERTA_MODEL,
        run_name=run_name,
        train_df=train_df,
        valid_df=valid_df,
        num_epochs=epochs,
        learning_rate=lr,
        weight_decay=wd,
        warmup_ratio=warmup,
        batch_train=ROBERTA_BATCH_TRAIN,
        batch_eval=ROBERTA_BATCH_EVAL,
        head_ratio=HEAD_RATIO,
        use_early_stopping=True,
        early_stopping_patience=2,
        save_model=False,  # On ne sauvegarde pas les checkpoints intermédiaires
    )
    grid_results.append({
        "run":      run_name,
        "epochs":   epochs,
        "lr":       lr,
        "warmup":   warmup,
        "wd":       wd,
        **{m: round(res[m],4) for m in ["accuracy","precision","recall","f1"]},
        "probs_valid": res["probs_valid"],
        "pred_valid":  res["pred_valid"],
    })

grid_df = pd.DataFrame(grid_results).sort_values("f1", ascending=False).reset_index(drop=True)
print("\n=== Résultats grid search Phase 2 ===")
display(grid_df[["run","epochs","lr","warmup","wd","accuracy","precision","recall","f1"]].head(10))


######################################################################
  roberta_grid_e2_lr2e-05_w0.06_wd0.01
  epochs=2 | lr=2e-05 | wd=0.01 | warmup=0.06
######################################################################


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


  roberta_grid_e4_lr3e-05_w0.1_wd0.1
              precision    recall  f1-score   support

           N     0.9735    0.9200    0.9460       200
           P     0.9242    0.9750    0.9489       200

    accuracy                         0.9475       400
   macro avg     0.9489    0.9475    0.9475       400
weighted avg     0.9489    0.9475    0.9475       400


=== Résultats grid search Phase 2 ===


,run,epochs,lr,warmup,wd,accuracy,precision,recall,f1
0,roberta_grid_e3_lr2e-05_w0.06_wd0.01,3,0.00002,0.06,0.01,0.9525,0.9249,0.985,0.9540
1,roberta_grid_e3_lr3e-05_w0.1_wd0.01,3,0.00003,0.10,0.01,0.9500,0.9286,0.975,0.9512
2,roberta_grid_e2_lr3e-05_w0.1_wd0.1,2,0.00003,0.10,0.10,0.9475,0.9202,0.980,0.9492
3,roberta_grid_e4_lr2e-05_w0.1_wd0.01,4,0.00002,0.10,0.01,0.9475,0.9202,0.980,0.9492
4,roberta_grid_e4_lr3e-05_w0.1_wd0.1,4,0.00003,0.10,0.10,0.9475,0.9242,0.975,0.9489
5,roberta_grid_e3_lr3e-05_w0.06_wd0.1,3,0.00003,0.06,0.10,0.9475,0.9453,0.950,0.9476
6,roberta_grid_e4_lr3e-05_w0.06_wd0.01,4,0.00003,0.06,0.01,0.9450,0.9045,0.995,0.9476
7,roberta_grid_e3_lr2e-05_w0.1_wd0.01,3,0.00002,0.10,0.01,0.9450,0.9159,0.980,0.9469
8,roberta_grid_e4_lr3e-05_w0.1_wd0.01,4,0.00003,0.10,0.01,0.9450,0.9198,0.975,0.9466
9,roberta_grid_e3_lr3e-05_w0.1_wd0.1,3,0.00003,0.10,0.10,0.9450,0.9198,0.975,0.9466


In [16]:
# Meilleure config RoBERTa
best_grid = grid_df.iloc[0]
print("\nMeilleure config :")
print(f"  epochs={best_grid['epochs']} | lr={best_grid['lr']} | warmup={best_grid['warmup']} | wd={best_grid['wd']}")
print(f"  F1={best_grid['f1']:.4f} | Accuracy={best_grid['accuracy']:.4f}")

# Réentraîner avec la meilleure config pour sauvegarder
roberta_best_result = run_transformer(
    model_name=ROBERTA_MODEL,
    run_name="roberta_best_p2",
    train_df=train_df,
    valid_df=valid_df,
    num_epochs=int(best_grid["epochs"]),
    learning_rate=float(best_grid["lr"]),
    weight_decay=float(best_grid["wd"]),
    warmup_ratio=float(best_grid["warmup"]),
    batch_train=ROBERTA_BATCH_TRAIN,
    batch_eval=ROBERTA_BATCH_EVAL,
    head_ratio=HEAD_RATIO,
    use_early_stopping=True,
    save_model=True,
)
RESULTS["RoBERTa_best"] = roberta_best_result


Meilleure config :
  epochs=3 | lr=2e-05 | warmup=0.06 | wd=0.01
  F1=0.9540 | Accuracy=0.9525

######################################################################
  roberta_best_p2
  epochs=3 | lr=2e-05 | wd=0.01 | warmup=0.06
######################################################################


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


  roberta_best_p2
              precision    recall  f1-score   support

           N     0.9789    0.9300    0.9538       200
           P     0.9333    0.9800    0.9561       200

    accuracy                         0.9550       400
   macro avg     0.9561    0.9550    0.9550       400
weighted avg     0.9561    0.9550    0.9550       400



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Modèle sauvegardé : /content/drive/MyDrive/projet tal/movie8_outputs/roberta_best_p2_best_model


---
# PHASE 3 — DeBERTa-v3-base

DeBERTa-v3 utilise SentencePiece — la tokenisation head+tail est compatible.  
Attention : DeBERTa nécessite `sentencepiece` installé.

In [17]:
!pip install -q sentencepiece protobuf

In [24]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

def run_transformer(
    model_name,
    run_name,
    train_df,
    valid_df,
    num_epochs,
    learning_rate,
    weight_decay,
    warmup_ratio,
    batch_train,
    batch_eval,
    max_length=512,
    head_ratio=1.0,
    use_early_stopping=True,
    early_stopping_patience=2,
    save_model=True,
    fp16=False,          # <-- ajouté
    bf16=False,          # <-- ajouté
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_ds = build_dataset(train_df, tokenizer, max_length=max_length)
    valid_ds = build_dataset(valid_df, tokenizer, max_length=max_length)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,   # adapte si besoin
    )

    training_args = TrainingArguments(
        output_dir=f"./outputs/{run_name}",
        num_train_epochs=num_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        per_device_train_batch_size=batch_train,
        per_device_eval_batch_size=batch_eval,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        fp16=fp16,   # <-- transmis ici
        bf16=bf16,   # <-- transmis ici
    )

    callbacks = []
    if use_early_stopping:
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=early_stopping_patience
            )
        )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    trainer.train()

    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "model": model,
    }

    RESULTS["DeBERTa_base"] = deberta_result

In [26]:
models_to_compare = ["SVM", "RoBERTa_base", "RoBERTa_best", "DeBERTa_base"]

missing = [k for k in models_to_compare if k not in RESULTS]
print("Manquants :", missing)

summary_p3 = pd.DataFrame([
    {
        "modèle": k,
        **{m: round(v, 4) for m, v in RESULTS[k].items() if m in ["accuracy", "precision", "recall", "f1"]}
    }
    for k in models_to_compare
    if k in RESULTS
]).set_index("modèle")

display(summary_p3)

Manquants : ['DeBERTa_base']


,accuracy,precision,recall,f1
modèle,,,,
SVM,0.890,0.8750,0.910,0.8922
RoBERTa_base,0.930,0.9019,0.965,0.9324
RoBERTa_best,0.955,0.9333,0.980,0.9561


---
# PHASE 4 — Vote majoritaire pondéré (soft voting)

On combine SVM + RoBERTa_best + DeBERTa par moyenne pondérée des probabilités.  
Grid search sur les poids w1/w2/w3 — objectif : **maximiser F1**.

In [28]:
available_models = {
    "SVM": RESULTS["SVM"]["probs_valid"],
    "RoBERTa_best": RESULTS["RoBERTa_best"]["probs_valid"],
}

if "DeBERTa_base" in RESULTS:
    available_models["DeBERTa_base"] = RESULTS["DeBERTa_base"]["probs_valid"]

print("Modèles utilisés pour le voting :", list(available_models.keys()))

y_true_valid = valid_df["label"].values
WEIGHT_STEPS = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]

model_names = list(available_models.keys())
voting_rows = []

for weights in itertools.product(WEIGHT_STEPS, repeat=len(model_names)):
    if sum(weights) == 0:
        continue

    total = sum(weights)
    avg_probs = sum(
        w * available_models[name]
        for w, name in zip(weights, model_names)
    ) / total

    pred_ids = np.argmax(avg_probs, axis=1)
    pred_labels = np.array([id2label[int(x)] for x in pred_ids])

    row = {
        "accuracy":  accuracy_score(y_true_valid, pred_labels),
        "precision": precision_score(y_true_valid, pred_labels, pos_label="P"),
        "recall":    recall_score(y_true_valid, pred_labels, pos_label="P"),
        "f1":        f1_score(y_true_valid, pred_labels, pos_label="P"),
    }

    for name, w in zip(model_names, weights):
        row[f"w_{name}"] = w

    voting_rows.append(row)

voting_df = (
    pd.DataFrame(voting_rows)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("=== Top 10 combinaisons (soft voting) ===")
display(voting_df.head(10).round(4))

Modèles utilisés pour le voting : ['SVM', 'RoBERTa_best']
=== Top 10 combinaisons (soft voting) ===


,accuracy,precision,recall,f1,w_SVM,w_RoBERTa_best
0,0.9650,0.9471,0.985,0.9657,3.0,1.5
1,0.9650,0.9471,0.985,0.9657,2.0,1.0
2,0.9650,0.9471,0.985,0.9657,1.0,0.5
3,0.9625,0.9557,0.970,0.9628,3.0,1.0
4,0.9625,0.9557,0.970,0.9628,1.5,0.5
5,0.9600,0.9423,0.980,0.9608,3.0,2.0
6,0.9600,0.9423,0.980,0.9608,1.5,1.0
7,0.9575,0.9420,0.975,0.9582,2.0,1.5
8,0.9550,0.9333,0.980,0.9561,0.0,1.5
9,0.9550,0.9333,0.980,0.9561,0.0,0.5


In [30]:
best_vote = voting_df.iloc[0]

w_svm_best = best_vote["w_SVM"]
w_roberta_best = best_vote["w_RoBERTa_best"]
total_best = w_svm_best + w_roberta_best

print(f"\nMeilleurs poids : SVM={w_svm_best}  RoBERTa={w_roberta_best}")
print(f"F1 ensemble validation : {best_vote['f1']:.4f}")
print(f"Accuracy               : {best_vote['accuracy']:.4f}")

avg_probs_best = (
    w_svm_best * probs_svm +
    w_roberta_best * probs_roberta
) / total_best

valid_pred_ensemble = np.array([
    id2label[int(x)] for x in np.argmax(avg_probs_best, axis=1)
])

RESULTS["Ensemble_vote"] = print_report(
    y_true_valid,
    valid_pred_ensemble,
    "Ensemble soft voting (validation)"
)


Meilleurs poids : SVM=3.0  RoBERTa=1.5
F1 ensemble validation : 0.9657
Accuracy               : 0.9650

  Ensemble soft voting (validation)
              precision    recall  f1-score   support

           N     0.9844    0.9450    0.9643       200
           P     0.9471    0.9850    0.9657       200

    accuracy                         0.9650       400
   macro avg     0.9657    0.9650    0.9650       400
weighted avg     0.9657    0.9650    0.9650       400



### Tableau de bord final — toutes phases

In [31]:
all_keys = ["SVM", "RoBERTa_base", "RoBERTa_best", "DeBERTa_base", "Ensemble_vote"]
dashboard = pd.DataFrame([
    {"modèle": k, **{m: round(RESULTS[k][m],4)
                    for m in ["accuracy","precision","recall","f1"]
                    if m in RESULTS[k]}}
    for k in all_keys if k in RESULTS
]).set_index("modèle")

print("\n" + "="*60)
print("  TABLEAU DE BORD FINAL — validation")
print("="*60)
display(dashboard)

best_model_name = dashboard["f1"].idxmax()
print(f"\nMeilleur modèle sur validation : {best_model_name}")
print(f"F1 = {dashboard.loc[best_model_name,'f1']:.4f}")


  TABLEAU DE BORD FINAL — validation


,accuracy,precision,recall,f1
modèle,,,,
SVM,0.890,0.8750,0.910,0.8922
RoBERTa_base,0.930,0.9019,0.965,0.9324
RoBERTa_best,0.955,0.9333,0.980,0.9561
Ensemble_vote,0.965,0.9471,0.985,0.9657



Meilleur modèle sur validation : Ensemble_vote
F1 = 0.9657


---
# Génération des soumissions test

4 fichiers de soumission :
1. `submission_svm.csv`
2. `submission_roberta_best.csv`
3. `submission_deberta.csv`
4. `submission_ensemble_vote.csv` ← à soumettre en priorité si meilleur F1

In [32]:
def predict_transformer_on_test(
    trainer, tokenizer, test_df,
    max_length=MAX_LENGTH, head_ratio=HEAD_RATIO
):
    hf_test = Dataset.from_pandas(test_df[["text"]].copy())
    tokenize_fn = make_head_tail_tokenize_fn(tokenizer, max_length, head_ratio)
    tok_test = hf_test.map(tokenize_fn, batched=True, remove_columns=["text"])
    output = trainer.predict(tok_test)
    probs  = softmax_np(output.predictions)
    pred_labels = np.array([id2label[int(x)] for x in np.argmax(output.predictions, axis=1)])
    return pred_labels, probs


def train_full_transformer(
    model_name, run_name, df_full,
    num_epochs, learning_rate, weight_decay, warmup_ratio,
    batch_train, batch_eval,
    max_length=MAX_LENGTH, head_ratio=HEAD_RATIO,
):
    """Réentraîne sur tout le corpus (train+valid) pour la soumission finale."""
    set_seed()
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tmp = df_full[["text", "label"]].copy()
    tmp["label"] = tmp["label"].map(label2id).astype(int)
    hf_full = Dataset.from_pandas(tmp.rename(columns={"label": "label"}))

    tokenize_fn = make_head_tail_tokenize_fn(tokenizer, max_length, head_ratio)
    tok_full = hf_full.map(tokenize_fn, batched=True, remove_columns=["text"])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"{run_name}_full_ckpt"),
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_train,
        per_device_eval_batch_size=batch_eval,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=tok_full,
        data_collator=data_collator,
    )
    trainer.train()
    return trainer, tokenizer


print("Fonctions de soumission définies")

Fonctions de soumission définies


In [33]:
if MAKE_TEST_SUBMISSIONS and test_df is not None:

    print("\n[1/4] Soumission SVM...")
    full_svm = build_svm()
    full_svm.fit(df["text"], df["label"])
    test_pred_svm  = full_svm.predict(test_df["text"])
    test_scores_svm = full_svm.decision_function(test_df["text"])
    test_probs_svm  = softmax_np(np.column_stack([-test_scores_svm, test_scores_svm]))
    save_submission(test_pred_svm, OUTPUT_DIR / "submission_svm.csv")

    print("\n[2/4] Soumission RoBERTa best...")
    roberta_full_trainer, roberta_full_tok = train_full_transformer(
        model_name=ROBERTA_MODEL,
        run_name="roberta_best_full",
        df_full=df,
        num_epochs=int(best_vote["w_roberta"] and best_grid["epochs"] or ROBERTA_BASE_EPOCHS),
        learning_rate=float(best_grid["lr"]),
        weight_decay=float(best_grid["wd"]),
        warmup_ratio=float(best_grid["warmup"]),
        batch_train=ROBERTA_BATCH_TRAIN,
        batch_eval=ROBERTA_BATCH_EVAL,
    )
    test_pred_roberta, test_probs_roberta = predict_transformer_on_test(
        roberta_full_trainer, roberta_full_tok, test_df
    )
    save_submission(test_pred_roberta, OUTPUT_DIR / "submission_roberta_best.csv")
    del roberta_full_trainer; gc.collect(); torch.cuda.empty_cache()

    print("\n[3/4] Soumission DeBERTa...")
    deberta_full_trainer, deberta_full_tok = train_full_transformer(
        model_name=DEBERTA_MODEL,
        run_name="deberta_full",
        df_full=df,
        num_epochs=DEBERTA_BASE_EPOCHS,
        learning_rate=DEBERTA_BASE_LR,
        weight_decay=DEBERTA_WEIGHT_DECAY,
        warmup_ratio=0.1,
        batch_train=DEBERTA_BATCH_TRAIN,
        batch_eval=DEBERTA_BATCH_EVAL,
    )
    test_pred_deberta, test_probs_deberta = predict_transformer_on_test(
        deberta_full_trainer, deberta_full_tok, test_df
    )
    save_submission(test_pred_deberta, OUTPUT_DIR / "submission_deberta.csv")
    del deberta_full_trainer; gc.collect(); torch.cuda.empty_cache()

    print("\n[4/4] Soumission Ensemble vote pondéré...")
    total_best = w_svm_best + w_roberta_best + w_deberta_best
    test_avg_probs = (
        w_svm_best     * test_probs_svm +
        w_roberta_best * test_probs_roberta +
        w_deberta_best * test_probs_deberta
    ) / total_best
    test_pred_ensemble = np.array([
        id2label[int(x)] for x in np.argmax(test_avg_probs, axis=1)
    ])
    save_submission(test_pred_ensemble, OUTPUT_DIR / "submission_ensemble_vote.csv")

    print("\n✅ 4 fichiers de soumission générés dans :", OUTPUT_DIR)
else:
    print("MAKE_TEST_SUBMISSIONS=False ou fichier test introuvable.")

MAKE_TEST_SUBMISSIONS=False ou fichier test introuvable.


---
## Résumé final & stratégie de soumission

Voici l'ordre de priorité de soumission :

| Priorité | Fichier | Quand le choisir |
|---|---|---|
| 1 | `submission_ensemble_vote.csv` | Si F1 ensemble > F1 RoBERTa ou DeBERTa seul |
| 2 | `submission_deberta.csv` | Si DeBERTa seul > ensemble (rare) |
| 3 | `submission_roberta_best.csv` | Si RoBERTa > DeBERTa (GPU limité) |
| 4 | `submission_svm.csv` | Backup uniquement |

**Règle d'or** : soumets d'abord le modèle avec le meilleur F1 **sur validation**, pas accuracy.

In [34]:
print("\n" + "="*60)
print("  RÉCAPITULATIF FINAL")
print("="*60)
display(dashboard)
print(f"\nMeilleur sur validation : {best_model_name}  F1={dashboard.loc[best_model_name,'f1']:.4f}")
print("\nFichiers générés :")
for f in sorted(OUTPUT_DIR.glob("submission_*.csv")):
    print(" -", f.name)


  RÉCAPITULATIF FINAL


,accuracy,precision,recall,f1
modèle,,,,
SVM,0.890,0.8750,0.910,0.8922
RoBERTa_base,0.930,0.9019,0.965,0.9324
RoBERTa_best,0.955,0.9333,0.980,0.9561
Ensemble_vote,0.965,0.9471,0.985,0.9657



Meilleur sur validation : Ensemble_vote  F1=0.9657

Fichiers générés :
